In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

# 1. 定义数据路径
drive_data_folder = '/content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet'
file_path = os.path.join(drive_data_folder, 'NHANES_Merged_Data.csv')

print(" 正在读取合并后的原始数据...")
df = pd.read_csv(file_path)
print(f" 原始数据维度: {df.shape}")

# ==========================================
# 第一步：重命名变量
# ==========================================
rename_dict = {
    'SEQN': 'ID',
    'RIAGENDR': 'Gender',                 # 1: 男, 2: 女
    'RIDAGEYR': 'Age',                    # 年龄
    'DMDEDUC2': 'Education',              # 教育程度
    'INDFMPIR': 'Income_Poverty_Ratio',   # 收入贫困比 (越高越富)
    'DMDMARTL': 'Marital_Status',         # 婚姻状况
    'DPQ010': 'PHQ9_1', 'DPQ020': 'PHQ9_2', 'DPQ030': 'PHQ9_3',
    'DPQ040': 'PHQ9_4', 'DPQ050': 'PHQ9_5', 'DPQ060': 'PHQ9_6',
    'DPQ070': 'PHQ9_7', 'DPQ080': 'PHQ9_8', 'DPQ090': 'PHQ9_9',
    'SLD012': 'Sleep_Hours',              # 睡眠时长
    'PAD680': 'Sedentary_Mins',           # 久坐时间 (分钟)
    'BMXBMI': 'BMI',                      # 体质指数
    'MCQ220': 'Cancer_History',           # 是否得过癌症
    'MCQ160C': 'Heart_Disease_History',   # 是否有冠心病
    'SMQ020': 'Smoking_Status',           # 是否抽烟超100根
    'ALQ130': 'Alcohol_Drinks_Per_Day'    # 过去12个月平均每天喝酒杯数
}

df.rename(columns=rename_dict, inplace=True)
print("列名重命名完成！")

# ==========================================
# 第二步：将“拒绝回答”或“不知道”替换为空值(NaN)
# ==========================================
# NHANES 官方手册规定：
# 对于 1位数的分类题，7=拒绝，9=不知道
# 对于 2位数的分类题，77=拒绝，99=不知道
# 对于 大数字题(如久坐时间)，7777=拒绝，9999=不知道

# 1. 处理抑郁症问卷 (0-3是有效得分，7和9是无效)
phq9_cols = [f'PHQ9_{i}' for i in range(1, 10)]
df[phq9_cols] = df[phq9_cols].replace({7: np.nan, 9: np.nan})

# 2. 处理分类协变量的异常值
df['Education'] = df['Education'].replace({7: np.nan, 9: np.nan})
df['Marital_Status'] = df['Marital_Status'].replace({77: np.nan, 99: np.nan})
df['Smoking_Status'] = df['Smoking_Status'].replace({7: np.nan, 9: np.nan})
df['Cancer_History'] = df['Cancer_History'].replace({7: np.nan, 9: np.nan})
df['Heart_Disease_History'] = df['Heart_Disease_History'].replace({7: np.nan, 9: np.nan})

# 3. 处理连续变量的异常值
# 每天喝酒杯数: 777和999是无效的
df['Alcohol_Drinks_Per_Day'] = df['Alcohol_Drinks_Per_Day'].replace({777: np.nan, 999: np.nan})
# 久坐时间(分钟): 7777和9999是无效的
df['Sedentary_Mins'] = df['Sedentary_Mins'].replace({7777: np.nan, 9999: np.nan})

print("'拒绝回答/不知道' 的异常值已成功替换为 NaN！")


# ==========================================
# 第三步：计算抑郁症得分 (Target Variable Y)
# ==========================================
# 严格标准：如果 9 个问题中哪怕有一个缺失，我们就不计算他的总分，记为 NaN
df['PHQ9_Score'] = df[phq9_cols].sum(axis=1, skipna=False)

# 衍生一个分类标签：是否患有抑郁症 (通常 PHQ-9 >= 10 被视为有临床意义的抑郁)
# 我们把它留作备用，因果推断通常可以用连续的得分做 Y，也可以用分类标签做 Y
df['Is_Depressed'] = (df['PHQ9_Score'] >= 10).astype(float) # 1为抑郁，0为健康，NaN保留
# 为了让之前的 NaN 不变成 0，我们恢复它的 NaN 身份
df.loc[df['PHQ9_Score'].isna(), 'Is_Depressed'] = np.nan

print("抑郁症得分 (PHQ9_Score) 计算完成！")


# ==========================================
# 第四步：清洗无效样本
# ==========================================
# 目标变量 (Y) 缺失的样本对我们毫无意义，直接删除整行
initial_rows = df.shape[0]
df.dropna(subset=['PHQ9_Score'], inplace=True)
dropped_rows = initial_rows - df.shape[0]

print(f"删除了 {dropped_rows} 个没有完成抑郁症问卷的受访者。")
print(f"第一步清洗完成！当前保留了 {df.shape[0]} 个有效样本。")


# ==========================================
# 保存清洗后的数据
# ==========================================
save_path = os.path.join(drive_data_folder, 'NHANES_Cleaned_Step1.csv')
df.to_csv(save_path, index=False)
print(f"数据已保存至: {save_path}")

# 展示前 5 行
display(df[['ID', 'Age', 'Gender', 'Sleep_Hours', 'Sedentary_Mins', 'PHQ9_Score', 'Is_Depressed']].head())

 正在读取合并后的原始数据...
 原始数据维度: (9254, 22)
列名重命名完成！
'拒绝回答/不知道' 的异常值已成功替换为 NaN！
抑郁症得分 (PHQ9_Score) 计算完成！
删除了 4186 个没有完成抑郁症问卷的受访者。
第一步清洗完成！当前保留了 5068 个有效样本。
数据已保存至: /content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet/NHANES_Cleaned_Step1.csv


,ID,Age,Gender,Sleep_Hours,Sedentary_Mins,PHQ9_Score,Is_Depressed
2,93705.0,66.0,2.0,8.0,300.0,4.857845e-78,0.0
3,93706.0,18.0,1.0,10.5,240.0,4.857845e-78,0.0
5,93708.0,66.0,2.0,8.0,120.0,4.857845e-78,0.0
8,93711.0,56.0,1.0,7.0,420.0,2.000000e+00,0.0
9,93712.0,18.0,1.0,7.5,120.0,1.000000e+00,0.0
